# Subcontractor Counts By Week

This notebook reads the wide billing output and builds a week-by-week count table using `Date_Delivered_to_VA`.

- Rows: `Subcontractor`
- Columns: custom weeks derived from `Date_Delivered_to_VA`
- Week definition: Saturday through Friday


In [ ]:
import csv
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path


In [ ]:
def resolve_input_csv_path() -> Path:
    candidates = [
        Path("output/HITL_billing_wide.csv"),
        Path("tmp_notebook_check/output/HITL_billing_wide.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def saturday_week_label(delivered_date: datetime) -> str:
    days_since_saturday = (delivered_date.weekday() - 5) % 7
    week_start = delivered_date - timedelta(days=days_since_saturday)
    week_end = week_start + timedelta(days=6)
    return f"{week_start:%Y-%m-%d} to {week_end:%Y-%m-%d}"


INPUT_CSV_PATH = resolve_input_csv_path()
INPUT_CSV_PATH


In [ ]:
counts_by_subcontractor = defaultdict(lambda: defaultdict(int))
weeks = set()

with INPUT_CSV_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        delivered_date = (row.get("Date_Delivered_to_VA") or "").strip()
        if not delivered_date:
            continue

        subcontractor = (row.get("Subcontractor") or "").strip() or "N/A"
        parsed_date = datetime.strptime(delivered_date, "%Y-%m-%d")
        week_label = saturday_week_label(parsed_date)

        counts_by_subcontractor[subcontractor][week_label] += 1
        weeks.add(week_label)

sorted_weeks = sorted(weeks)
preferred_subcontractor_order = ["XBP", "IRM", "TSS", "VA", "N/A"]
sorted_subcontractors = [
    subcontractor
    for subcontractor in preferred_subcontractor_order
    if subcontractor in counts_by_subcontractor
] + sorted(
    subcontractor
    for subcontractor in counts_by_subcontractor
    if subcontractor not in preferred_subcontractor_order
)

counts_table = []
for subcontractor in sorted_subcontractors:
    result_row = {"Subcontractor": subcontractor}
    for week_label in sorted_weeks:
        result_row[week_label] = counts_by_subcontractor[subcontractor].get(week_label, 0)
    counts_table.append(result_row)

print(f"Input CSV: {INPUT_CSV_PATH.resolve()}")
print(f"Weeks found: {sorted_weeks}")
print(f"Subcontractors found: {sorted_subcontractors}")


In [ ]:
counts_table
